# Load Data

In [5]:
import pandas as pd
from pathlib import Path

RAW = Path("../datasheet/raw")

files = ["assessments", "courses", "studentAssessment", "studentInfo", "studentRegistration", "vle"]
df = {name: pd.read_csv(RAW / f"{name}.csv") for name in files}

for name, d in df.items():
    print(f"\n=== {name} | {d.shape[0]:,} baris x {d.shape[1]} kolom")
    na = d.isna().sum()
    na = na[na > 0]
    print(na.to_string() if len(na) else "tidak ada missing value")


=== assessments | 206 baris x 6 kolom
date    11

=== courses | 22 baris x 3 kolom
tidak ada missing value

=== studentAssessment | 173,912 baris x 5 kolom
score    173

=== studentInfo | 32,593 baris x 12 kolom
imd_band    1111

=== studentRegistration | 32,593 baris x 5 kolom
date_registration         45
date_unregistration    22521

=== vle | 6,364 baris x 6 kolom
week_from    5243
week_to      5243


In [6]:
asm = df["assessments"]
sa = df["studentAssessment"]

print("=== Jenis assessment:")
print(asm["assessment_type"].value_counts(), "\n")

print("=== Assessment yang date-nya kosong:")
print(asm[asm["date"].isna()]["assessment_type"].value_counts(), "\n")

print("=== Contoh isi studentAssessment:")
print(sa.head(), "\n")

print("=== is_banked:")
print(sa["is_banked"].value_counts())

=== Jenis assessment:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64 

=== Assessment yang date-nya kosong:
assessment_type
Exam    11
Name: count, dtype: int64 

=== Contoh isi studentAssessment:
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
3           1752       32885              26          0   69.0
4           1752       38053              19          0   79.0 

=== is_banked:
is_banked
0    172003
1      1909
Name: count, dtype: int64


## Cleaning 1: Buang baris banked
Baris `is_banked = 1` adalah nilai yang dibawa dari periode sebelumnya
(mahasiswa mengulang), bukan tugas yang benar-benar dikerjakan.

In [ ]:
sa_clean = sa[sa["is_banked"] == 0]
print(len(sa), "→", len(sa_clean))

173912 → 172003


In [10]:
data = sa_clean.merge(asm, on="id_assessment", how="left")

print(data.shape)
print(data["assessment_type"].value_counts())
data.head()

(172003, 10)
assessment_type
TMA     97088
CMA     69956
Exam     4959
Name: count, dtype: int64


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


## Cleaning 2: Buang Exam + cek kualitas data
Exam dijadwalkan pada hari yang sama untuk semua mahasiswa,
jadi tidak mencerminkan durasi pengerjaan tugas.

In [11]:
# 1. Buang Exam
data = data[data["assessment_type"] != "Exam"]
print("Jumlah baris:", len(data))
print(data["assessment_type"].value_counts(), "\n")

# 2. Cek: masih ada deadline kosong?
print("Deadline kosong:", data["date"].isna().sum())

# 3. Cek: ada data dobel?
dobel = data.duplicated(subset=["id_student", "id_assessment"]).sum()
print("Baris dobel:", dobel, "\n")

# 4. Hitung selisih hari: deadline - tanggal kumpul
data["hari_sebelum_deadline"] = data["date"] - data["date_submitted"]
data["hari_sebelum_deadline"].describe()

Jumlah baris: 167044
assessment_type
TMA    97088
CMA    69956
Name: count, dtype: int64 

Deadline kosong: 0
Baris dobel: 0 



count    167044.000000
mean         15.885318
std          45.236433
min        -372.000000
25%          -2.000000
50%           1.000000
75%           5.000000
max         246.000000
Name: hari_sebelum_deadline, dtype: float64